In [240]:
import pandas as pd
import numpy as np

In [241]:
df_wasde = pd.read_csv("..//data/oce-wasde-report-data-2026-09.csv")
df_wasde_hist = pd.read_csv("../data/wasde_processed_all_historicals.csv")


WASDE

In [242]:
df_wasde_hist.columns

Index(['Report Date', 'Day', 'Month', 'Year', 'Value', 'commodity', 'country',
       'snd_item', 'wheat_class', 'crop', 'table', 'crop year',
       'month offset'],
      dtype='str')

In [243]:
display_items = [
    "Area Planted",
    "Area Harvested",
    "Yield per Harvested Acre",
    "Beginning Stocks",
    "Production",
    "Imports",
    "Supply, Total",
    "Feed and Residual",
    "Food, Seed & Industrial",
    "Ethanol & by-products",
    "Exports",
    "Use, Total",
    "Ending Stocks"
]

In [244]:
filtered_years = df_wasde[
    (df_wasde["Commodity"] == "Corn") &
    (df_wasde["Region"] == "United States")&
    (df_wasde["Attribute"].isin(display_items))
].copy()



In [245]:
df_wasde_corn = filtered_years.pivot_table(
    index="Attribute",
    columns="MarketYear",
    values="Value",
)


In [287]:

df_wasde_hist

selected_month = 9
selected_commodity = "soybeans"
selected_country = "united states"

yearly_wasde = df_wasde_hist[
    (df_wasde_hist["commodity"] == selected_commodity) &
    (df_wasde_hist["country"] == selected_country) &
    (df_wasde_hist["crop"] == "nc")&
    (df_wasde_hist["Month"] == selected_month)&
    (df_wasde_hist["table"] == "us")
].copy()


In [288]:
common_map = {
    "beginning stocks": "Beginning Stocks",
    "production": "Production",
    "imports": "Imports",
    "supply total": "Total Supply",
    "exports": "Exports",
    "use total": "Total Use",
    "ending stocks": "Ending Stocks",
    "stock to use": "Stocks-to-Use",
    "planted area": "Area Planted",
    "harvested area": "Area Harvested",
    "yield": "Yield"
}

commodity_specific_map = {

    "corn": {
        "domestic total": "Domestic Total",
        "ethanol": "Ethanol & by-products",
        "feed residual": "Feed and Residual",
        "food seed industrial": "FSI",
    },

    "wheat": {
        "domestic total": "Domestic Total",
        "ethanol": "Ethanol & by-products",
        "feed residual": "Feed and Residual",
        "food": "Food",
        "seed": "Seed",
    },

    "soybeans": {
        "crushing": "Crushing",
        "residual": "Residual",
        "seed": "Seed",
    },
}

commodity_map = {
    **common_map,
    **commodity_specific_map[selected_commodity]
}

In [292]:
historicals = yearly_wasde.pivot(
    index="snd_item",
    columns="crop year",
    values="Value"
).copy()

display_order = [
    key for key in common_map
    if key in historicals.index
]

demand_items = [
    key for key in commodity_specific_map[selected_commodity]
    if key in historicals.index
]

insert_position = display_order.index("use total")

display_order[insert_position:insert_position] = demand_items

historicals = (
    historicals
    .loc[display_order]
    .rename(index=commodity_map)
)

display(historicals.round(2))

crop year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
snd_item,,,,,,,,,,,,,,,,,
Beginning Stocks,150.00,225.00,130.00,125.00,130.00,210.00,195.00,345.00,395.0,1005.00,575.0,175.00,240.00,250.00,340.00,330.00,325.00
Production,3483.00,3085.00,2634.00,3149.00,3913.00,3935.00,4201.00,4431.00,4693.0,3633.00,4313.0,4374.00,4378.00,4146.00,4586.00,4301.00,4535.00
Imports,10.00,15.00,20.00,15.00,15.00,30.00,30.00,25.00,25.0,20.00,15.0,25.00,15.00,30.00,15.00,20.00,25.00
Total Supply,3643.00,3325.00,2785.00,3289.00,4058.00,4175.00,4426.00,4801.00,5113.0,4658.00,4903.0,4574.00,4633.00,4426.00,4941.00,4651.00,4885.00
Exports,1485.00,1415.00,1055.00,1370.00,1700.00,1725.00,1985.00,2250.00,2060.0,1775.00,2125.0,2090.00,2085.00,1790.00,1850.00,1685.00,1685.00
Crushing,1650.00,1635.00,1500.00,1655.00,1770.00,1870.00,1950.00,1940.00,2070.0,2115.00,2180.0,2180.00,2225.00,2290.00,2425.00,2555.00,2780.00
Residual,70.00,23.00,25.00,28.00,22.00,38.00,31.00,35.00,34.0,32.00,38.0,14.00,21.00,25.00,38.00,37.00,38.00
Seed,88.00,88.00,89.00,87.00,92.00,92.00,95.00,101.00,103.0,96.00,100.0,104.00,102.00,101.00,78.00,73.00,73.00
Total Use,3293.00,3161.00,2670.00,3140.00,3583.00,3725.00,4061.00,4326.00,4268.0,4018.00,4442.0,4389.00,4433.00,4206.00,4391.00,4351.00,4575.00


In [290]:
selected_month = 9
selected_country = "united states"

commodities = ["corn", "soybeans", "wheat"]


def get_wasde_matrix(
    df,
    commodity,
    month,
    country="united states",
):
    return df[
        (df["commodity"] == commodity) &
        (df["country"] == country) &
        (df["crop"] == "nc") &
        (df["Month"] == month) &
        (df["table"] == "us")
    ].copy()


wasde_matrices = {
    commodity: get_wasde_matrix(
        df_wasde_hist,
        commodity=commodity,
        month=selected_month,
        country=selected_country,
    )
    for commodity in commodities
}

In [296]:
selected_commodity = "wheat"

commodity_matrix = wasde_matrices[selected_commodity]

commodity_map = {
    **common_map,
    **commodity_specific_map[selected_commodity]
}

wasde_historical = commodity_matrix.pivot(
    index="snd_item",
    columns="crop year",
    values="Value"
).copy()

display_order = [
    key for key in common_map
    if key in wasde_historical.index
]

demand_items = [
    key for key in commodity_specific_map[selected_commodity]
    if key in wasde_historical.index
]

insert_position = display_order.index("use total")

display_order[insert_position:insert_position] = demand_items

wasde_historical = (
    wasde_historical
    .loc[display_order]
    .rename(index=commodity_map)
)

display(wasde_historical.round(2))

crop year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
snd_item,,,,,,,,,,,,,,,,,
Beginning Stocks,973.00,861.00,743.00,718.00,590.00,753.00,981.00,1184.00,1100.00,1072.00,1044.00,844.0,660.00,580.00,702.00,851.00,920.00
Production,2265.00,2077.00,2268.00,2114.00,2030.00,2136.00,2321.00,1739.00,1877.00,1980.00,1838.00,1697.0,1783.00,1734.00,1982.00,1927.00,1531.00
Imports,100.00,110.00,130.00,140.00,170.00,125.00,115.00,150.00,135.00,135.00,130.00,135.0,110.00,130.00,105.00,120.00,140.00
Total Supply,3338.00,3047.00,3141.00,2972.00,2789.00,3014.00,3417.00,3074.00,3112.00,3187.00,3011.00,2676.0,2553.00,2444.00,2789.00,2898.00,2591.00
Exports,1250.00,1025.00,1200.00,1100.00,900.00,900.00,950.00,975.00,1025.00,975.00,975.00,875.0,825.00,700.00,825.00,900.00,775.00
Domestic Total,1186.00,1262.00,1243.00,1311.00,1191.00,1239.00,1367.00,1166.00,1152.00,1198.00,1111.00,1186.0,1118.00,1129.00,1136.00,1154.00,1099.00
Feed and Residual,170.00,240.00,220.00,280.00,155.00,200.00,330.00,150.00,120.00,170.00,90.00,160.0,80.00,90.00,110.00,120.00,80.00
Food,940.00,940.00,950.00,958.00,960.00,967.00,968.00,950.00,970.00,960.00,960.00,964.0,970.00,974.00,964.00,972.00,960.00
Seed,76.00,82.00,73.00,73.00,76.00,72.00,69.00,66.00,62.00,68.00,61.00,62.0,68.00,65.00,62.00,62.00,59.00


In [313]:
selected_commodity = "corn"
selected_metric = "exports"

commodity_seasonality = df_wasde_hist[
    (df_wasde_hist["commodity"] == selected_commodity) &
    (df_wasde_hist["country"] == "united states") &
    (df_wasde_hist["crop"] == "nc") &
    (df_wasde_hist["table"] == "us") &
    (df_wasde_hist["snd_item"] == "exports")
].copy()

commodity_seasonality["Report Date"] = pd.to_datetime(
    commodity_seasonality["Report Date"]
)

commodity_seasonality["WASDE Month"] = (
    commodity_seasonality["Report Date"].dt.month
)

seasonality_matrix = commodity_seasonality.pivot(
    index="crop year",
    columns="WASDE Month",
    values="Value"
).copy()

month_map = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Aug",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec"
}

seasonality_matrix = seasonality_matrix.rename(columns=month_map)

month_order = [
    "May", "Jun", "Jul", "Aug", "Sep", "Oct",
    "Nov", "Dec", "Jan", "Feb", "Mar", "Apr"
]

seasonality_matrix = seasonality_matrix.reindex(
    columns=month_order
)

display(seasonality_matrix)

WASDE Month,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Apr
crop year,,,,,,,,,,,,
2010,2000.0,2000.0,1950.0,2050.0,2100.0,2000.0,1950.0,1950.0,1950.0,1950.0,1950.0,1950.0
2011,1800.0,1800.0,1900.0,1750.0,1650.0,1600.0,1600.0,1600.0,1650.0,1700.0,1700.0,1700.0
2012,1900.0,1900.0,1600.0,1300.0,1250.0,1150.0,1150.0,1150.0,950.0,900.0,825.0,800.0
2013,1300.0,1300.0,1250.0,1225.0,1225.0,NaN,1400.0,1450.0,1450.0,1600.0,1625.0,1750.0
2014,1700.0,1700.0,1700.0,1725.0,1750.0,1750.0,1750.0,1750.0,1750.0,1750.0,1800.0,1800.0
2015,1900.0,1900.0,1875.0,1850.0,1850.0,1850.0,1800.0,1750.0,1700.0,1650.0,1650.0,1650.0
2016,1900.0,1950.0,2050.0,2175.0,2175.0,2225.0,2225.0,2225.0,2225.0,2225.0,2225.0,2225.0
2017,1875.0,1875.0,1875.0,1850.0,1850.0,1850.0,1925.0,1925.0,1925.0,2050.0,2225.0,2225.0
2018,2100.0,2100.0,2225.0,2350.0,2400.0,2475.0,2450.0,2450.0,NaN,2450.0,2375.0,2300.0


In [316]:
# Create display labels separately
crop_year_labels = {
    year: f"{year}/{str(year + 1)[-2:]}"
    for year in seasonality_matrix.index
}

# Display table with formatted crop-year labels
display_matrix = seasonality_matrix.rename(index=crop_year_labels)

display(display_matrix)

#corn exports with marketing years

WASDE Month,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Apr
crop year,,,,,,,,,,,,
2010/11,2000.0,2000.0,1950.0,2050.0,2100.0,2000.0,1950.0,1950.0,1950.0,1950.0,1950.0,1950.0
2011/12,1800.0,1800.0,1900.0,1750.0,1650.0,1600.0,1600.0,1600.0,1650.0,1700.0,1700.0,1700.0
2012/13,1900.0,1900.0,1600.0,1300.0,1250.0,1150.0,1150.0,1150.0,950.0,900.0,825.0,800.0
2013/14,1300.0,1300.0,1250.0,1225.0,1225.0,NaN,1400.0,1450.0,1450.0,1600.0,1625.0,1750.0
2014/15,1700.0,1700.0,1700.0,1725.0,1750.0,1750.0,1750.0,1750.0,1750.0,1750.0,1800.0,1800.0
2015/16,1900.0,1900.0,1875.0,1850.0,1850.0,1850.0,1800.0,1750.0,1700.0,1650.0,1650.0,1650.0
2016/17,1900.0,1950.0,2050.0,2175.0,2175.0,2225.0,2225.0,2225.0,2225.0,2225.0,2225.0,2225.0
2017/18,1875.0,1875.0,1875.0,1850.0,1850.0,1850.0,1925.0,1925.0,1925.0,2050.0,2225.0,2225.0
2018/19,2100.0,2100.0,2225.0,2350.0,2400.0,2475.0,2450.0,2450.0,NaN,2450.0,2375.0,2300.0


In [315]:
import pandas as pd
import plotly.graph_objects as go


# ============================================================
# LAST 5 CROP YEARS
# ============================================================

last_5_years = [2022, 2023, 2024, 2025, 2026]


# ============================================================
# CREATE FIGURE
# ============================================================

fig = go.Figure()


for crop_year in last_5_years:

    # Skip if year doesn't exist
    if crop_year not in seasonality_matrix.index:
        continue

    values = pd.to_numeric(
        seasonality_matrix.loc[crop_year, month_order],
        errors="coerce"
    )

    # Display format: 2026 -> 2026/27
    display_year = f"{crop_year}/{str(crop_year + 1)[-2:]}"


    # ========================================================
    # CURRENT YEAR
    # ========================================================

    if crop_year == 2026:

        fig.add_trace(
            go.Scatter(
                x=month_order,
                y=values,
                mode="lines+markers",

                name=display_year,

                line=dict(width=4),

                marker=dict(size=8),

                hovertemplate=(
                    "<b>%{x}</b><br>"
                    f"Crop Year: {display_year}<br>"
                    "Exports: %{y:,.0f}"
                    "<extra></extra>"
                )
            )
        )


    # ========================================================
    # HISTORICAL YEARS
    # ========================================================

    else:

        fig.add_trace(
            go.Scatter(
                x=month_order,
                y=values,
                mode="lines",

                name=display_year,

                line=dict(width=2),

                opacity=0.45,

                hovertemplate=(
                    "<b>%{x}</b><br>"
                    f"Crop Year: {display_year}<br>"
                    "Exports: %{y:,.0f}"
                    "<extra></extra>"
                )
            )
        )


# ============================================================
# LAYOUT
# ============================================================

fig.update_layout(
    title="US Corn Exports – WASDE Revision History",

    xaxis=dict(
        title="WASDE Month",
        categoryorder="array",
        categoryarray=month_order
    ),

    yaxis=dict(
        title="Exports"
    ),

    template="plotly_white",

    hovermode="x unified",

    height=600,

    legend=dict(
        title="Crop Year"
    )
)


fig.show()